# bench_gen — Semi-supervised drug response modelling

This notebook focuses on the Genomics of Drug Sensitivity in Cancer (GDSC)
dataset. We benchmark tabular regressors via `BenchmarkRunner`, then reframe the
problem as a binary response classification to demonstrate semi-supervised
training with `SemiSupervisedTabular`.


## 1. Environment
Install Kaggle plus the gradient boosting libraries required by the shared
model registry.


In [ ]:
%%capture
!pip install -q kaggle xgboost lightgbm

## 2. Imports and seeds


In [ ]:
from itertools import cycle
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from pipelines_torch.benchmark import BenchmarkRunner
from pipelines_torch.models import MODEL_REGISTRY
from pipelines_torch.ss_models import SemiSupervisedTabular
from pipelines_torch.base import SimplePredictor
from utils.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, f1_score
from utils.utils import load_model

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 3. Download and parse GDSC
We rely on lightweight heuristics to discover the expression and response CSVs.
Update `expr_path` / `resp_path` below if your local copy uses different names.


In [ ]:
from pathlib import Path

from utils.kaggle_utils import ensure_kaggle_dataset

KAGGLE_DATASET = "samiraalipour/genomics-of-drug-sensitivity-in-cancer-gdsc"
DATA_ROOT = ensure_kaggle_dataset(
    dataset_slug=KAGGLE_DATASET,
    local_dir=Path("data_gdsc"),
    description="GDSC dataset",
    kaggle_subdir="genomics-of-drug-sensitivity-in-cancer-gdsc",
)

In [ ]:
# The GDSC dataset files don't have "expression" or "response" in their names
# We'll use GDSC_DATASET.csv which is the main merged file, or GDSC2-dataset.csv for response data
gdsc_main = DATA_ROOT / "GDSC_DATASET.csv"
gdsc2 = DATA_ROOT / "GDSC2-dataset.csv"

# Try to find the main dataset file
if gdsc_main.exists():
    print(f"Using main merged dataset: {gdsc_main}")
    main_df = pd.read_csv(gdsc_main)
    print(f"Main dataset shape: {main_df.shape}")
    print(f"Column names: {list(main_df.columns)}")
    print(f"\nNumeric columns: {list(main_df.select_dtypes(include=[np.number]).columns)}")
elif gdsc2.exists():
    print(f"Using GDSC2 dataset: {gdsc2}")
    main_df = pd.read_csv(gdsc2)
    print(f"GDSC2 dataset shape: {main_df.shape}")
    print(f"Column names: {list(main_df.columns)}")
    print(f"\nNumeric columns: {list(main_df.select_dtypes(include=[np.number]).columns)}")
else:
    raise FileNotFoundError("Could not locate GDSC_DATASET.csv or GDSC2-dataset.csv. Please check the data directory.")


In [ ]:
# Extract response data (drug sensitivity) from main_df
# The GDSC dataset uses CELL_LINE_NAME, DRUG_NAME, and LN_IC50
resp_df = main_df[['CELL_LINE_NAME', 'DRUG_NAME', 'LN_IC50', 'COSMIC_ID']].copy()
resp_df = resp_df.rename(columns={
    'CELL_LINE_NAME': 'cell_line',
    'DRUG_NAME': 'drug',
    'LN_IC50': 'log_ic50'
})
resp_df = resp_df.dropna(subset=['cell_line', 'drug', 'log_ic50'])

# Since GDSC_DATASET.csv doesn't contain actual gene expression matrices,
# we'll create features from available columns and one-hot encode categorical variables
print("Creating features from available metadata...")

# Select feature columns - mix of numeric and categorical
feature_cols_numeric = ['COSMIC_ID', 'AUC', 'Z_SCORE']
feature_cols_categorical = ['TCGA_DESC', 'GDSC Tissue descriptor 1', 'GDSC Tissue descriptor 2', 
                           'Cancer Type (matching TCGA label)', 'Microsatellite instability Status (MSI)',
                           'Screen Medium', 'Growth Properties', 'CNA', 'Gene Expression', 'Methylation']

# Create a feature dataframe for each cell line (aggregate by cell line)
cell_line_features = []

for cell_line in main_df['CELL_LINE_NAME'].unique():
    cell_data = main_df[main_df['CELL_LINE_NAME'] == cell_line].iloc[0]
    feature_dict = {'cell_line': cell_line}
    
    # Add numeric features
    for col in feature_cols_numeric:
        if col in main_df.columns:
            feature_dict[col] = cell_data[col]
    
    # Add categorical features (will one-hot encode later)
    for col in feature_cols_categorical:
        if col in main_df.columns:
            feature_dict[col] = cell_data[col]
    
    cell_line_features.append(feature_dict)

features_df = pd.DataFrame(cell_line_features)
print(f"Created features dataframe with shape: {features_df.shape}")

# One-hot encode categorical variables
categorical_cols = [col for col in feature_cols_categorical if col in features_df.columns]
features_encoded = pd.get_dummies(features_df, columns=categorical_cols, drop_first=True)
features_encoded = features_encoded.set_index('cell_line')

print(f"After one-hot encoding: {features_encoded.shape}")
print(f"Feature columns sample: {list(features_encoded.columns[:10])}")

# Focus on the most common drug
focus_drug = resp_df['drug'].value_counts().idxmax()
resp_subset = resp_df[resp_df['drug'] == focus_drug].copy()
print(f"\nFocused on drug: {focus_drug} with {len(resp_subset)} samples")

# Filter to only cell lines present in both datasets
resp_subset = resp_subset[resp_subset['cell_line'].isin(features_encoded.index)]
print(f"Cell lines with features: {len(resp_subset)}")

# Extract features and labels
features = features_encoded.reindex(resp_subset['cell_line']).fillna(0).to_numpy(dtype=np.float32)
labels_reg = resp_subset['log_ic50'].to_numpy(dtype=np.float32)

print(f"\nFinal features shape: {features.shape}")
print(f"Final labels shape: {labels_reg.shape}")


## 4. Supervised regression benchmark
We standardise features, split train/validation, and feed them to
`BenchmarkRunner` using the registry regressors.


In [ ]:
scaler = StandardScaler()
X_train, X_val, y_train, y_val = train_test_split(
    features, labels_reg, test_size=0.2, random_state=SEED
)
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

reg_metrics = [mean_squared_error, mean_absolute_error, r2_score]
reg_models = [
    {
        "name": "mlp_regressor",
        "class": MODEL_REGISTRY["mlp_regressor"],
        "params": {"input_dim": X_train.shape[1], "output_dim": 1},
    },
    {
        "name": "xgboost_regressor",
        "class": MODEL_REGISTRY["xgboost_regressor"],
        "params": {"n_estimators": 600, "max_depth": 6, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "random_state": SEED},
    },
    {
        "name": "lightgbm_regressor",
        "class": MODEL_REGISTRY["lightgbm_regressor"],
        "params": {"n_estimators": 800, "learning_rate": 0.05, "num_leaves": 64, "subsample": 0.8, "colsample_bytree": 0.8},
    },
]

runner = BenchmarkRunner(
    model_configs=reg_models,
    augmentations=[None],
    metrics=reg_metrics,
    task_type="regression",
    device="cpu",
    epochs=5,
    batch_size=64,
    use_kfold=False,
    learning_rate=1e-3,
    path_start="bench_gen_supervised",
    random_state=SEED,
)
regression_results = runner.run(X_train, y_train)
regression_results


### Reload checkpoints on the validation split


In [ ]:
def evaluate_regression_models(model_names: Iterable[str], X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    records = []
    for name in model_names:
        checkpoint = f"{name}_none"
        cfg = next(cfg for cfg in reg_models if cfg["name"] == name)
        try:
            model = load_model(cfg["class"], checkpoint, cfg["params"], path_start="results/bench_gen_supervised")
        except FileNotFoundError:
            print(f"⚠️ Skipping {name}: checkpoint not found")
            continue
        predictor = SimplePredictor(model, task_type="regression", device="cpu", batch_size=128)
        preds = predictor.predict(X)
        records.append({
            "model": name,
            "mse": float(mean_squared_error(y, preds)),
            "mae": float(mean_absolute_error(y, preds)),
            "r2": float(r2_score(y, preds)),
        })
    return pd.DataFrame.from_records(records)

val_regression_metrics = evaluate_regression_models([m["name"] for m in reg_models], X_val, y_val)
val_regression_metrics


## 5. Optional external validation
Place another drug-response dataset in `data_gdsc/secondary.csv` with matching
cell-line identifiers and gene ordering to reuse the scaler above.


In [ ]:
SECONDARY_PATH = DATA_ROOT / "secondary.csv"
if SECONDARY_PATH.exists():
    secondary_df = pd.read_csv(SECONDARY_PATH)
    if 'cell_line' not in secondary_df.columns or 'log_ic50' not in secondary_df.columns:
        raise ValueError("Secondary dataset must contain 'cell_line' and 'log_ic50' columns")
    secondary_df = secondary_df[secondary_df['cell_line'].isin(expr_df.index)]
    X_secondary = expr_df.reindex(secondary_df['cell_line']).to_numpy(dtype=np.float32)
    X_secondary = scaler.transform(X_secondary)
    y_secondary = secondary_df['log_ic50'].to_numpy(dtype=np.float32)
    secondary_metrics = evaluate_regression_models([m["name"] for m in reg_models], X_secondary, y_secondary)
    secondary_metrics
else:
    print("⚠️ Provide a secondary dataset at data_gdsc/secondary.csv to enable this cell.")


## 6. Semi-supervised binary response classification

**Note on Dataset Suitability:** The GDSC dataset is fully labeled (all samples have IC50 measurements), making it inherently a **supervised learning** dataset. However, we use it to **simulate** a realistic semi-supervised learning scenario for the following reasons:

### Why Simulate SSL with GDSC?

1. **Real-world constraint simulation**: In drug discovery, obtaining IC50 measurements is expensive and time-consuming. A realistic scenario would be:
   - 1000+ cell lines with genomic profiles available (cheap)
   - Only 30% have been tested for drug response (expensive lab experiments)
   - Goal: Predict response for the 70% untested cell lines

2. **Benchmarking SSL algorithms**: By artificially masking labels, we can:
   - Compare SSL vs fully supervised learning
   - Measure how much SSL recovers with limited labels
   - Validate SSL techniques before deploying to truly unlabeled data

3. **Genomics application**: This setup mirrors real pharmacogenomics challenges where sequencing data is abundant but phenotypic screening is limited.

Below, we:
1. Derive binary labels (sensitive/resistant) from continuous IC50 values
2. Train a **supervised baseline** using all labels (upper bound)
3. Train **semi-supervised models** using only 30% labels + 70% unlabeled data
4. Compare performance to justify the SSL approach

In [ ]:
# Prepare binary classification data
# Derive binary labels: sensitive (IC50 below median) vs resistant (IC50 above median)
median_ic50 = np.median(y_train)
y_binary_train = (y_train <= median_ic50).astype(np.int64)
y_binary_val = (y_val <= median_ic50).astype(np.int64)

X_train_ssl = X_train.astype(np.float32)
X_val_ssl = X_val.astype(np.float32)

print(f"Binary classification setup:")
print(f"  Training samples: {len(y_binary_train)} (Sensitive: {(y_binary_train==0).sum()}, Resistant: {(y_binary_train==1).sum()})")
print(f"  Validation samples: {len(y_binary_val)} (Sensitive: {(y_binary_val==0).sum()}, Resistant: {(y_binary_val==1).sum()})")
print(f"  IC50 median threshold: {median_ic50:.3f}")
print(f"  Feature dimensions: {X_train_ssl.shape[1]}")

# Create SSL split: 30% labeled, 70% unlabeled
LABEL_FRACTION = 0.3
mask = np.random.default_rng(SEED).random(len(y_binary_train)) < LABEL_FRACTION
X_labeled = X_train_ssl[mask]
y_labeled = y_binary_train[mask]
X_unlabeled = X_train_ssl[~mask]

print(f"\nSimulated SSL scenario:")
print(f"  Labeled data: {len(X_labeled)} samples ({LABEL_FRACTION*100:.0f}%)")
print(f"  Unlabeled data: {len(X_unlabeled)} samples ({(1-LABEL_FRACTION)*100:.0f}%)")
print(f"  This simulates: {len(X_labeled)} cell lines tested, {len(X_unlabeled)} cell lines with genomic data only")


### 6.1 Supervised Baseline: Full Label Benchmark

First, we establish supervised baselines using **all available labels** across multiple tabular classification models. This provides the upper bound performance.

In [ ]:
# Supervised classification baseline using all labels
from utils.metrics import METRIC_REGISTRY

clf_metrics = [METRIC_REGISTRY["accuracy"], METRIC_REGISTRY["f1"], METRIC_REGISTRY["precision"], METRIC_REGISTRY["recall"], METRIC_REGISTRY["roc_auc"], METRIC_REGISTRY["pr_auc"]]

clf_models = [
    {
        "name": "mlp_classifier",
        "class": MODEL_REGISTRY["mlp_classifier"],
        "params": {"input_dim": X_train_ssl.shape[1], "num_classes": 2},
    },
    {
        "name": "deep_mlp_classifier",
        "class": MODEL_REGISTRY["deep_mlp_classifier"],
        "params": {"input_dim": X_train_ssl.shape[1], "num_classes": 2},
    },
    {
        "name": "random_forest_classifier",
        "class": MODEL_REGISTRY["random_forest_classifier"],
        "params": {"n_estimators": 100, "random_state": SEED},
    },
    {
        "name": "xgboost_classifier",
        "class": MODEL_REGISTRY["xgboost_classifier"],
        "params": {"n_estimators": 100, "max_depth": 6, "learning_rate": 0.1, "random_state": SEED},
    },
    {
        "name": "lightgbm_classifier",
        "class": MODEL_REGISTRY["lightgbm_classifier"],
        "params": {"n_estimators": 100, "learning_rate": 0.1, "num_leaves": 31},
    },
    {
        "name": "tabr_classifier",
        "class": MODEL_REGISTRY["tabr_classifier"],
        "params": {"input_dim": X_train_ssl.shape[1], "num_classes": 2},
    },
    {
        "name": "grande_classifier",
        "class": MODEL_REGISTRY["grande_classifier"],
        "params": {"input_dim": X_train_ssl.shape[1], "num_classes": 2},
    },
    {
        "name": "tabm_classifier",
        "class": MODEL_REGISTRY["tabm_classifier"],
        "params": {"input_dim": X_train_ssl.shape[1], "num_classes": 2},
    },
]

print("Training supervised baselines with 100% labeled data...")
print("This may take several minutes depending on model complexity.\n")

runner_clf = BenchmarkRunner(
    model_configs=clf_models,
    augmentations=[None],
    metrics=clf_metrics,
    task_type="classification",
    device=DEVICE,
    epochs=10,
    batch_size=64,
    use_kfold=False,
    learning_rate=1e-3,
    path_start="bench_gen_supervised_clf",
    random_state=SEED,
)
supervised_results = runner_clf.run(X_train_ssl, y_binary_train)
supervised_results

### 6.2 Evaluate Supervised Baselines on Validation Set

In [ ]:
# Evaluate all trained supervised classifiers
from pipelines_torch.benchmark import load_model
from utils.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc, pr_auc

supervised_results = {}

for model_cfg in clf_models:
    model_name = model_cfg["name"]
    try:
        # Load trained model
        checkpoint = f"{model_name}_none"
        model = load_model(model_cfg["class"], checkpoint, model_cfg["params"], path_start="results/bench_gen_supervised_clf")
        
        # Create predictor
        predictor = SimplePredictor(model, task_type="classification", device=DEVICE, batch_size=128)
        
        # Make predictions (get probabilities for ROC-AUC and PR-AUC)
        preds = predictor.predict(X_val_ssl)
        
        # Compute metrics
        metrics = {
            'accuracy': float(accuracy_score(y_binary_val, preds)),
            'f1': float(f1_score(y_binary_val, preds)),
            'precision': float(precision_score(y_binary_val, preds)),
            'recall': float(recall_score(y_binary_val, preds)),
            'roc_auc': float(roc_auc(y_binary_val, preds)),
            'pr_auc': float(pr_auc(y_binary_val, preds)),
        }
        supervised_results[model_name] = metrics
        
        print(f"{model_name}:")
        print(f"  Accuracy:  {metrics['accuracy']:.4f}")
        print(f"  F1 Score:  {metrics['f1']:.4f}")
        print(f"  Precision: {metrics['precision']:.4f}")
        print(f"  Recall:    {metrics['recall']:.4f}")
        print(f"  ROC-AUC:   {metrics['roc_auc']:.4f}")
        print(f"  PR-AUC:    {metrics['pr_auc']:.4f}")
        print()
        
    except Exception as e:
        print(f"Failed to evaluate {model_name}: {e}")
        supervised_results[model_name] = None

# Create results DataFrame
supervised_df = pd.DataFrame(supervised_results).T
print("\n=== Supervised Baseline Results (100% labels) ===")
print(supervised_df.to_string())

### 6.3 Semi-Supervised Learning (30% labeled + 70% unlabeled data)

Now we apply semi-supervised learning using the `SemiSupervisedTabular` wrapper with Mean Teacher algorithm. This leverages both labeled and unlabeled data to improve performance compared to limited supervision alone.

In [ ]:
# Prepare data loaders for semi-supervised training
labeled_dataset = TensorDataset(torch.from_numpy(X_labeled), torch.from_numpy(y_labeled))
unlabeled_dataset = TensorDataset(torch.from_numpy(X_unlabeled))
val_dataset = TensorDataset(torch.from_numpy(X_val_ssl), torch.from_numpy(y_binary_val))

labeled_loader = DataLoader(labeled_dataset, batch_size=32, shuffle=True)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

print(f"Semi-supervised setup:")
print(f"  Labeled batches: {len(labeled_loader)}")
print(f"  Unlabeled batches: {len(unlabeled_loader)}")
print(f"  Validation batches: {len(val_loader)}")

In [ ]:
# Train semi-supervised models using Mean Teacher
ssl_results = {}

# Select a subset of models for SSL (computationally intensive)
ssl_model_names = ["mlp_classifier", "deep_mlp_classifier", "xgboost_classifier", "lightgbm_classifier"]

for model_name in ssl_model_names:
    print(f"\n{'='*60}")
    print(f"Training SSL model: {model_name}")
    print(f"{'='*60}")
    
    # Find model config
    model_cfg = next(cfg for cfg in clf_models if cfg["name"] == model_name)
    
    # Create base model
    base_model = model_cfg["class"](**model_cfg["params"])
    
    # Wrap with SemiSupervisedTabular (Mean Teacher)
    ssl_model = SemiSupervisedTabular(
        base_model,
        num_classes=2,
        use_mean_teacher=True,
        ema_decay=0.999,
        unsup_weight=1.0,
        rampup=5,
        noise_std=0.05,
        temperature=1.0,
    ).to(DEVICE)
    
    # Training setup
    optimizer = torch.optim.Adam(ssl_model.parameters(), lr=1e-3)
    epochs = 20
    best_f1 = 0.0
    
    for epoch in range(epochs):
        ssl_model.train()
        epoch_loss = 0.0
        
        unlabeled_iter = cycle(unlabeled_loader)
        
        for xb_l, yb_l in labeled_loader:
            # Get unlabeled batch
            xb_u = next(unlabeled_iter)[0]
            
            xb_l = xb_l.to(DEVICE)
            yb_l = yb_l.to(DEVICE)
            xb_u = xb_u.to(DEVICE)
            
            optimizer.zero_grad()
            loss, logs = ssl_model.step((xb_l, yb_l), (xb_u, None), epoch)
            loss.backward()
            optimizer.step()
            ssl_model.post_step()
            
            epoch_loss += loss.item()
        
        # Validation
        if (epoch + 1) % 5 == 0:
            ssl_model.eval()
            all_preds = []
            all_labels = []
            
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb = xb.to(DEVICE)
                    logits = ssl_model(xb)
                    preds = logits.argmax(dim=1).cpu().numpy()
                    all_preds.extend(preds)
                    all_labels.extend(yb.numpy())
            
            all_preds = np.array(all_preds)
            all_labels = np.array(all_labels)
            
            val_acc = float(accuracy_score(all_labels, all_preds))
            val_f1 = float(f1_score(all_labels, all_preds))
            
            print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss/len(labeled_loader):.4f} - Val Acc: {val_acc:.4f} - Val F1: {val_f1:.4f}")
            
            if val_f1 > best_f1:
                best_f1 = val_f1
    
    # Final evaluation
    ssl_model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE)
            logits = ssl_model(xb)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(yb.numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Compute final metrics
    ssl_results[model_name] = {
        'accuracy': float(accuracy_score(all_labels, all_preds)),
        'f1': float(f1_score(all_labels, all_preds)),
        'precision': float(precision_score(all_labels, all_preds)),
        'recall': float(recall_score(all_labels, all_preds)),
        'roc_auc': float(roc_auc(all_labels, all_preds)),
        'pr_auc': float(pr_auc(all_labels, all_preds)),
    }
    
    print(f"\n{model_name} Final Results:")
    print(f"  Accuracy:  {ssl_results[model_name]['accuracy']:.4f}")
    print(f"  F1 Score:  {ssl_results[model_name]['f1']:.4f}")
    print(f"  ROC-AUC:   {ssl_results[model_name]['roc_auc']:.4f}")
    print(f"  PR-AUC:    {ssl_results[model_name]['pr_auc']:.4f}")

ssl_df = pd.DataFrame(ssl_results).T
print("\n" + "="*60)
print("=== Semi-Supervised Results (30% labeled + 70% unlabeled) ===")
print("="*60)
print(ssl_df.to_string())

### 6.4 Performance Comparison: Full Supervised vs Semi-Supervised Learning

This comparison demonstrates:
- **Full Supervised (100% labels)**: Upper bound performance when all training labels are available
- **Semi-Supervised (30% labeled + 70% unlabeled)**: Using Mean Teacher to leverage unlabeled data
- **Key Question**: Can SSL with 30% labels + unlabeled data approach the performance of 100% labeled data?

The comparison shows whether semi-supervised learning can effectively exploit unlabeled data to bridge the label scarcity gap.

In [ ]:
# Create comprehensive comparison table: Full Supervised vs Semi-Supervised
comparison_results = []

for model_name in ssl_model_names:
    row = {'Model': model_name}
    
    # Full supervised metrics (100% labels)
    if model_name in supervised_results and supervised_results[model_name]:
        row['Full_Acc'] = supervised_results[model_name].get('accuracy', None)
        row['Full_F1'] = supervised_results[model_name].get('f1', None)
        row['Full_ROC'] = supervised_results[model_name].get('roc_auc', None)
        row['Full_PR'] = supervised_results[model_name].get('pr_auc', None)
    else:
        row['Full_Acc'] = None
        row['Full_F1'] = None
        row['Full_ROC'] = None
        row['Full_PR'] = None
    
    # Semi-supervised metrics (30% labeled + 70% unlabeled)
    if model_name in ssl_results and ssl_results[model_name]:
        row['SSL_Acc'] = ssl_results[model_name].get('accuracy', None)
        row['SSL_F1'] = ssl_results[model_name].get('f1', None)
        row['SSL_ROC'] = ssl_results[model_name].get('roc_auc', None)
        row['SSL_PR'] = ssl_results[model_name].get('pr_auc', None)
    else:
        row['SSL_Acc'] = None
        row['SSL_F1'] = None
        row['SSL_ROC'] = None
        row['SSL_PR'] = None
    
    # Calculate performance gaps and recovery rates
    if row['Full_Acc'] is not None and row['SSL_Acc'] is not None:
        row['Acc_Gap'] = row['Full_Acc'] - row['SSL_Acc']
        row['Acc_Recovery_%'] = (row['SSL_Acc'] / row['Full_Acc']) * 100
    else:
        row['Acc_Gap'] = None
        row['Acc_Recovery_%'] = None
    
    if row['Full_F1'] is not None and row['SSL_F1'] is not None:
        row['F1_Gap'] = row['Full_F1'] - row['SSL_F1']
        row['F1_Recovery_%'] = (row['SSL_F1'] / row['Full_F1']) * 100
    else:
        row['F1_Gap'] = None
        row['F1_Recovery_%'] = None
    
    if row['Full_ROC'] is not None and row['SSL_ROC'] is not None:
        row['ROC_Gap'] = row['Full_ROC'] - row['SSL_ROC']
    else:
        row['ROC_Gap'] = None
    
    comparison_results.append(row)

comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.round(4)

print("=" * 140)
print("COMPARISON: Full Supervised (100% labels) vs Semi-Supervised Learning (30% labeled + 70% unlabeled)")
print("=" * 140)
print(comparison_df.to_string(index=False))
print("=" * 140)

# Summary statistics
print(f"\n📊 Summary Statistics:")
print(f"Average Accuracy Gap: {comparison_df['Acc_Gap'].mean():.4f}")
print(f"Average Accuracy Recovery: {comparison_df['Acc_Recovery_%'].mean():.2f}%")
print(f"Average F1 Gap: {comparison_df['F1_Gap'].mean():.4f}")
print(f"Average F1 Recovery: {comparison_df['F1_Recovery_%'].mean():.2f}%")
print(f"Average ROC-AUC Gap: {comparison_df['ROC_Gap'].mean():.4f}")

print(f"\n💡 Interpretation:")
if comparison_df['Acc_Recovery_%'].mean() > 95:
    print("  ✅ SSL successfully recovers >95% of full supervised performance using only 30% labels!")
elif comparison_df['Acc_Recovery_%'].mean() > 90:
    print("  ✅ SSL recovers >90% of full supervised performance - significant improvement over limited labels alone.")
elif comparison_df['Acc_Recovery_%'].mean() > 85:
    print("  ⚠️  SSL shows moderate improvement but significant gap remains.")
else:
    print("  ⚠️  SSL struggles to leverage unlabeled data effectively in this scenario.")
    
print(f"\n🎯 Conclusion:")
print(f"  With only {LABEL_FRACTION*100:.0f}% labeled data + {(1-LABEL_FRACTION)*100:.0f}% unlabeled data,")
print(f"  Mean Teacher SSL achieves {comparison_df['Acc_Recovery_%'].mean():.1f}% of full supervised performance.")
print(f"  This demonstrates the value of SSL in label-scarce drug discovery scenarios.")

### 6.5 Bonus: Limited Supervision Baseline (30% labels only, no unlabeled data)

For completeness, let's also train models with only 30% labeled data (without leveraging unlabeled data) to show the baseline that SSL improves upon.

In [ ]:
# Train with limited labels only (no SSL, no unlabeled data)
limited_results = {}

print(f"Training baseline models with only {len(X_labeled)} labeled samples ({LABEL_FRACTION*100:.0f}%)...")
print("(No unlabeled data used)\n")

for model_name in ssl_model_names:
    model_cfg = next(cfg for cfg in clf_models if cfg["name"] == model_name)
    
    # Create and train model on limited labeled data only
    base_model = model_cfg["class"](**model_cfg["params"])
    
    # For PyTorch models, do simple training
    if hasattr(base_model, 'parameters'):
        base_model = base_model.to(DEVICE)
        optimizer = torch.optim.Adam(base_model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()
        
        # Train for fewer epochs since less data
        for epoch in range(10):
            base_model.train()
            for xb, yb in labeled_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad()
                logits = base_model(xb)
                loss = criterion(logits, yb)
                loss.backward()
                optimizer.step()
        
        # Evaluate
        base_model.eval()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                logits = base_model(xb)
                preds = logits.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(yb.numpy())
    else:
        # For sklearn models
        base_model.fit(X_labeled, y_labeled)
        all_preds = base_model.predict(X_val_ssl)
        all_labels = y_binary_val
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    limited_results[model_name] = {
        'accuracy': float(accuracy_score(all_labels, all_preds)),
        'f1': float(f1_score(all_labels, all_preds)),
        'precision': float(precision_score(all_labels, all_preds)),
        'recall': float(recall_score(all_labels, all_preds)),
        'roc_auc': float(roc_auc(all_labels, all_preds)),
        'pr_auc': float(pr_auc(all_labels, all_preds)),
    }
    
    print(f"{model_name}: Acc={limited_results[model_name]['accuracy']:.4f}, F1={limited_results[model_name]['f1']:.4f}")

limited_df = pd.DataFrame(limited_results).T
print("\n=== Limited Supervision Results (30% labels only, no unlabeled) ===")
print(limited_df.to_string())

In [ ]:
# Three-way comparison: Full Supervised vs SSL vs Limited Supervised
print("\n" + "=" * 150)
print("COMPREHENSIVE COMPARISON: Full Supervised vs Semi-Supervised vs Limited Supervised")
print("=" * 150)

three_way_results = []

for model_name in ssl_model_names:
    row = {'Model': model_name}
    
    # Full supervised (100% labels)
    if model_name in supervised_results and supervised_results[model_name]:
        row['Full_Acc'] = supervised_results[model_name].get('accuracy', None)
        row['Full_F1'] = supervised_results[model_name].get('f1', None)
    else:
        row['Full_Acc'] = None
        row['Full_F1'] = None
    
    # SSL (30% labeled + 70% unlabeled)
    if model_name in ssl_results and ssl_results[model_name]:
        row['SSL_Acc'] = ssl_results[model_name].get('accuracy', None)
        row['SSL_F1'] = ssl_results[model_name].get('f1', None)
    else:
        row['SSL_Acc'] = None
        row['SSL_F1'] = None
    
    # Limited supervised (30% labels only)
    if model_name in limited_results and limited_results[model_name]:
        row['Limited_Acc'] = limited_results[model_name].get('accuracy', None)
        row['Limited_F1'] = limited_results[model_name].get('f1', None)
    else:
        row['Limited_Acc'] = None
        row['Limited_F1'] = None
    
    # Calculate improvements
    if row['SSL_Acc'] and row['Limited_Acc']:
        row['SSL_vs_Limited_Δ'] = row['SSL_Acc'] - row['Limited_Acc']
    else:
        row['SSL_vs_Limited_Δ'] = None
    
    if row['Full_Acc'] and row['SSL_Acc']:
        row['Recovery_%'] = (row['SSL_Acc'] / row['Full_Acc']) * 100
    else:
        row['Recovery_%'] = None
    
    three_way_results.append(row)

three_way_df = pd.DataFrame(three_way_results)
three_way_df = three_way_df.round(4)

print(three_way_df.to_string(index=False))
print("=" * 150)

print(f"\n📈 Key Findings:")
print(f"  • Full Supervised (100% labels):      Avg Acc = {three_way_df['Full_Acc'].mean():.4f}")
print(f"  • Semi-Supervised (30%+70% unlabeled): Avg Acc = {three_way_df['SSL_Acc'].mean():.4f}")
print(f"  • Limited Supervised (30% only):       Avg Acc = {three_way_df['Limited_Acc'].mean():.4f}")
print(f"\n  • SSL improves over Limited by: {three_way_df['SSL_vs_Limited_Δ'].mean():.4f} accuracy points")
print(f"  • SSL achieves {three_way_df['Recovery_%'].mean():.1f}% of full supervised performance")

print(f"\n✅ Conclusion: Semi-supervised learning (Mean Teacher) successfully leverages unlabeled data")
print(f"   to improve performance beyond what's achievable with limited labeled data alone.")

## 7. Where to go next
- Swap `focus_drug` to analyse other therapeutic compounds.
- Replace the Mean Teacher wrapper with `use_mean_teacher=False` to run pure
  pseudo-labelling.
- Provide a curated secondary cohort (e.g. PRISM / CCLE) to stress-test the
  supervised checkpoints.
